# 順順行 實測資料探索

1. 把手機匯出的檔案放進 `analysis/data/`（可分子資料夾）
2. `cd analysis && uv sync --extra notebook && uv run jupyter lab`
3. 由上往下執行。改 `OVERRIDES` 可以重新調參，結果與手機用同一套演算法（`shared/params.default.json`）。

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

from sunsunbus import load_params, process_trip
from sunsunbus.io import find_trips, load_trip
from sunsunbus import stats, plots

DATA = Path('../data')
OUT = Path('../output/notebook'); (OUT / 'plots').mkdir(parents=True, exist_ok=True)

# 調參：只寫要改的欄位，例如 {'events': {'brakeThreshold': 1.2}, 'lowpassCutoffHz': 2.5}
OVERRIDES = {}
P = load_params(OVERRIDES)

bases = find_trips(DATA) or find_trips('../tests/fixtures')  # 還沒有實測資料時先用合成資料
print(len(bases), '趟')

In [ ]:
items = []
for b in bases:
    trip = load_trip(b)
    r = process_trip(trip.motion, trip.orientation, trip.gps, P)
    items.append((trip, r))
trips = stats.trips_table(items)
trips[['trip', 'route', 'placement', 'seat_row', 'duration_min', 'distance_km', 'sample_rate_hz', 'stable_ratio',
       'n_hard_brake', 'n_hard_accel', 'n_sharp_turn', 'n_high_jerk', 'smooth_score', 'orient_sources']]

## 單趟時間序列（灰底＝不穩定、色塊＝事件、紅色虛線＝乘客按「剛剛急煞了」）

In [ ]:
trip, r = items[0]
marks = [m['t'] for m in trip.meta.get('markers', []) if m.get('kind') == 'manual_brake']
path = OUT / 'plots' / f'{trip.name}.png'
plots.plot_trip(r, trip.name, path, P, marks)
display(Image(str(path)))

## 事件列表與事件率（路線 × 時段；未滿 10 趟 publishable = False）

In [ ]:
events = stats.events_table(items)
display(events.head(20))
stats.rates_by(trips, ['route', 'hour'])

## 分布：放置方式（支架／座椅／手持）與座位的差異

In [ ]:
plots.plot_distributions(items, OUT / 'dist_placement.png', 'placement')
plots.plot_distributions(items, OUT / 'dist_seat.png', 'seat_row')
display(Image(str(OUT / 'dist_placement.png')))
stats.distribution(items, 'placement')

## 事件地圖

In [ ]:
plots.events_map(items, OUT / 'events_map.html')
from IPython.display import IFrame
IFrame(str(OUT / 'events_map.html'), width='100%', height=480)

## 對應到路線站間（需要 TDX 路線資料）

```
TDX_CLIENT_ID=... TDX_CLIENT_SECRET=... uv run sunsunbus fetch-tdx 中山幹線 --out data/routes
```

In [ ]:
from sunsunbus import routes
shape_file = DATA / 'routes' / '中山幹線_Shape.json'
stops_file = DATA / 'routes' / '中山幹線_StopOfRoute.json'
if shape_file.exists() and not events.empty:
    shapes = routes.load_route_shapes(shape_file)
    shape = routes.pick_shape(shapes, [(g.lat, g.lon) for t, _ in items for g in t.gps])
    stops = routes.load_stops(stops_file, shape.direction) if stops_file.exists() else None
    ev = routes.assign_events(events, shape, stops)
    display(ev.groupby(['between_stops', 'label']).size().unstack(fill_value=0))
else:
    print('沒有路線資料，略過')

## 手機 vs 硬體感測盒

In [ ]:
from sunsunbus import hardware
box_file = DATA / 'box.csv'   # 改成實際的檔名
if box_file.exists():
    trip, phone_r = items[0]
    box = hardware.read_box_csv(box_file)
    taps = [m['t'] for m in trip.meta.get('markers', []) if m.get('kind') == 'sync_tap']
    al = hardware.align_box(trip.motion, box, taps[0] if taps else None)
    box_r = hardware.process_box(box, al['offset_ms'], trip.gps, P)
    plots.plot_phone_vs_box(phone_r, box_r, OUT / 'phone_vs_box.png')
    display(Image(str(OUT / 'phone_vs_box.png')))
    display(al, hardware.compare_phone_box(phone_r, box_r))
else:
    print('沒有感測盒資料，略過')